# Zero-layer attention-only transformer

logits = W_U W_E x  — the model can only learn bigram statistics.

First, we load the corpus; for this demo, we're using Alice in Wonderland, and we use a word-level tokenizer.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from src.data import load_corpus

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

corpus = load_corpus()
print(f"{len(corpus):,} tokens, vocab {corpus.vocab_size:,}, device {device}")

34,354 tokens, vocab 2,687, device cuda


Since we use a word-level tokenizer, we can retrieve the bigram statistics simply by checking the `[t, t+1]` pairs.

In [2]:
from src.data import TokenWindows

dataset = TokenWindows(corpus.ids, context_len=1, stride=1, device=device)

pairwise_counts = torch.zeros((corpus.vocab_size, corpus.vocab_size)).to(device)

for example in tqdm(dataset, desc="processing data..."): # type: ignore
    first, second = example[0].item(), example[1].item()
    pairwise_counts[first][second] += 1

# bigram frequencies = pairwise_counts row-wise summed
bigram_frequencies = pairwise_counts / torch.sum(pairwise_counts, dim=1, keepdim=True)
print(bigram_frequencies)

processing data...: 100%|██████████| 34353/34353 [00:00<00:00, 51374.28it/s]


tensor([[0.0000, 0.0140, 0.0820,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0188, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
       device='cuda:0')


Now we can write our demo 0-layer attention only transformer, which will be trained on the tokenized corpus and re-assessed.

In [3]:
class ZeroLayerAttentionOnlyTransformer(nn.Module):
    def __init__(self, vocab_dim: int, h_dim: int, device: str):
        super().__init__()
        self.W_E = nn.Embedding(vocab_dim, h_dim, device=device)
        self.W_U = nn.Linear(h_dim, vocab_dim, bias=False, device=device)

    def forward(self, x):
        return self.W_U(self.W_E(x))

H_DIM = 128
TRAIN_STEPS = 5000

toy_transformer = ZeroLayerAttentionOnlyTransformer(vocab_dim=corpus.vocab_size, h_dim=H_DIM, device=device)
opt = torch.optim.Adam(toy_transformer.parameters(), lr=1e-2)
ids = corpus.tensor(device)
x, y = ids[:-1], ids[1:]

losses=[]
for _ in tqdm(range(TRAIN_STEPS), desc="training..."):
    logits = toy_transformer(x)
    loss = F.cross_entropy(logits, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
print(f"Last 5 losses: {losses[-5:]}")

training...: 100%|██████████| 5000/5000 [00:24<00:00, 206.03it/s]

Last 5 losses: [2.9880242347717285, 2.988022804260254, 2.9880223274230957, 2.9880237579345703, 2.9880261421203613]


Note that these last five losses are super close to one another. That is because we are hard capped at the limit given by the entropy of the natural bigram distribution inside our text. Next, we construct $W^U W^E$ to see if it aligns with our bigram frequency matrix from earlier.

In [4]:
W_E = toy_transformer.W_E.weight.data.detach()
W_U = toy_transformer.W_U.weight.data.detach()

# get table, replace unseen stuff with -inf
recovered_frequency_table = F.log_softmax(W_E@W_U.T, dim=1)
seen = pairwise_counts > 0
recovered_frequency_table = recovered_frequency_table.where(seen, float('-inf'))
log_frequencies = torch.log(bigram_frequencies)
print(f"Recovered Table:\n{recovered_frequency_table}\nLog Table:{log_frequencies}")

diff = (recovered_frequency_table - log_frequencies)[seen].abs().mean().item()
print(f"Mean difference between recovered matrix and bigram statistic: [{diff}]")

Recovered Table:
tensor([[       -inf, -4.2645e+00, -2.5050e+00,  ...,        -inf,
                -inf,        -inf],
        [       -inf,        -inf,        -inf,  ...,        -inf,
                -inf,        -inf],
        [       -inf, -3.9723e+00,        -inf,  ...,        -inf,
                -inf,        -inf],
        ...,
        [       -inf,        -inf,        -inf,  ...,        -inf,
                -inf,        -inf],
        [-1.3113e-06,        -inf,        -inf,  ...,        -inf,
                -inf,        -inf],
        [       -inf,        -inf,        -inf,  ...,        -inf,
                -inf,        -inf]], device='cuda:0')
Log Table:tensor([[   -inf, -4.2681, -2.5011,  ...,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,  ...,    -inf,    -inf,    -inf],
        [   -inf, -3.9739,    -inf,  ...,    -inf,    -inf,    -inf],
        ...,
        [   -inf,    -inf,    -inf,  ...,    -inf,    -inf,    -inf],
        [ 0.0000,    -inf,    -

So we've done it! We successfully recovered the bigram statistics matrix from training with AdamW on a 0-layer attention-only transformer.